# SUPRA-JEPA sur Google Colab

Notebook pour preparer les donnees, pre-entrainer SUPRA-JEPA, lancer le fine-tuning binaire supraconducteur / non-supraconducteur, puis entrainer un modele de diffusion qui genere des candidats scores par le JEPA.

Avant de lancer : dans Colab, active `Runtime > Change runtime type > GPU` si disponible.


In [7]:
import os, sys, json, subprocess, textwrap
from pathlib import Path

print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch pas encore importe:', exc)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 1. Installer les dependances

Colab contient deja PyTorch. On installe surtout `matminer`, `mp-api`, `pymatgen`, `spglib` et `scikit-learn`.

In [8]:
!pip -q install matminer mp-api pymatgen spglib scikit-learn

uoyYNpcD7FkEcG6K9msPWeouVvjO17EV

## 2. Charger le projet

Par defaut, le notebook clone ton repo GitHub dans `/content/SUPRACONDUCTOR-JEPA`.

In [9]:
REPO_URL = 'https://github.com/Baptistecaille/SUPRACONDUCTOR-JEPA.git'
PROJECT_DIR = Path('/content/SUPRACONDUCTOR-JEPA')

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Projet introuvable: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Working directory:', Path.cwd())
!ls -la

Working directory: /content/SUPRACONDUCTOR-JEPA
total 648
drwxr-xr-x 8 root root   4096 Jun 26 09:37 .
drwxr-xr-x 1 root root   4096 Jun 26 09:37 ..
drwxr-xr-x 2 root root   4096 Jun 26 09:37 checkpoints
-rw-r--r-- 1 root root   3306 Jun 26 09:37 config.py
drwxr-xr-x 2 root root   4096 Jun 26 09:37 data
-rw-r--r-- 1 root root   9527 Jun 26 09:37 data.py
-rw-r--r-- 1 root root   9207 Jun 26 09:37 diffusion.py
-rw-r--r-- 1 root root   8942 Jun 26 09:37 finetune.py
-rw-r--r-- 1 root root   4491 Jun 26 09:37 generate_materials.py
drwxr-xr-x 8 root root   4096 Jun 26 09:37 .git
-rw-r--r-- 1 root root     88 Jun 26 09:37 main.py
-rw-r--r-- 1 root root  12366 Jun 26 09:37 model.py
drwxr-xr-x 2 root root   4096 Jun 26 09:37 notebooks
-rw-r--r-- 1 root root   4293 Jun 26 09:37 pretrain.py
drwxr-xr-x 2 root root   4096 Jun 26 09:37 __pycache__
-rw-r--r-- 1 root root    500 Jun 26 09:37 pyproject.toml
-rw-r--r-- 1 root root      5 Jun 26 09:37 .python-version
-rw-r--r-- 1 root root      0 Jun 26 

## 3. Parametres Colab

Ces valeurs sont petites pour valider le pipeline. Augmente-les ensuite pour un vrai run.

In [18]:
MAX_SC = 100
MAX_NONSC = 100

PRETRAIN_EPOCHS = 20
FINETUNE_EPOCHS = 80
DIFFUSION_EPOCHS = 500
BATCH_SIZE_PRETRAIN = 32
BATCH_SIZE_FINETUNE = 32
BATCH_SIZE_DIFFUSION = 32

N_GENERATED = 64
TOP_K = 20
TRAIN_DIFFUSION_ON_POSITIVES_ONLY = True

print({
    'MAX_SC': MAX_SC,
    'MAX_NONSC': MAX_NONSC,
    'PRETRAIN_EPOCHS': PRETRAIN_EPOCHS,
    'FINETUNE_EPOCHS': FINETUNE_EPOCHS,
    'DIFFUSION_EPOCHS': DIFFUSION_EPOCHS,
    'N_GENERATED': N_GENERATED,
    'TOP_K': TOP_K,
})


{'MAX_SC': 100, 'MAX_NONSC': 100, 'PRETRAIN_EPOCHS': 20, 'FINETUNE_EPOCHS': 80, 'DIFFUSION_EPOCHS': 500, 'N_GENERATED': 64, 'TOP_K': 20}


## 4. Preparer les donnees

Il faut une cle API Materials Project. Elle est demandee de maniere masquee.

In [11]:
from getpass import getpass

mp_api_key = getpass('Materials Project API key: ')
cmd = [
    sys.executable, '-m', 'scripts.prepare_data',
    '--mp-api-key', mp_api_key,
    '--max-sc', str(MAX_SC),
    '--max-nonsc', str(MAX_NONSC),
]
subprocess.run(cmd, check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'scripts.prepare_data', '--mp-api-key', 'uoyYNpcD7FkEcG6K9msPWeouVvjO17EV', '--max-sc', '100', '--max-nonsc', '100'], returncode=0)

In [12]:
from data import load_supercon_dataset

structures, labels = load_supercon_dataset('data')
print('Total:', len(labels))
print('SC:', sum(labels))
print('non-SC:', len(labels) - sum(labels))

Total: 200
SC: 100
non-SC: 100


## 5. Pretraining JEPA

On appelle les fonctions Python directement pour surcharger les hyperparametres sans modifier les fichiers du repo.

In [13]:
import torch
from torch.utils.data import DataLoader

from config import Config
from data import CrystalDataset
from model import SupraJEPA
from pretrain import pretrain

cfg = Config()
cfg.epochs_pretrain = PRETRAIN_EPOCHS
cfg.batch_size_pretrain = BATCH_SIZE_PRETRAIN
cfg.warmup_steps = 10
cfg.data_dir = 'data'

torch.manual_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

pretrain_ds = CrystalDataset(structures, [-1] * len(structures), cfg.max_atoms, cfg.mask_ratio)
pretrain_loader = DataLoader(pretrain_ds, batch_size=cfg.batch_size_pretrain, shuffle=True, num_workers=2)

model = SupraJEPA(cfg)
print('Parametres entrainables:', sum(p.numel() for p in model.parameters() if p.requires_grad))
model = pretrain(model, pretrain_loader, cfg, device)

Device: cuda
Parametres entrainables: 5991169


/content/SUPRACONDUCTOR-JEPA/model.py:111: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
/content/SUPRACONDUCTOR-JEPA/model.py:148: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


[Pretrain] Epoch 001/20 | loss=0.0065 | lr=7.00e-05
  ✓ checkpoint sauvegardé (loss=0.0065)
[Pretrain] Epoch 002/20 | loss=0.0036 | lr=9.98e-05
  ✓ checkpoint sauvegardé (loss=0.0036)
[Pretrain] Epoch 003/20 | loss=0.0024 | lr=9.82e-05
  ✓ checkpoint sauvegardé (loss=0.0024)
[Pretrain] Epoch 004/20 | loss=0.0019 | lr=9.53e-05
  ✓ checkpoint sauvegardé (loss=0.0019)
[Pretrain] Epoch 005/20 | loss=0.0016 | lr=9.11e-05
  ✓ checkpoint sauvegardé (loss=0.0016)
[Pretrain] Epoch 006/20 | loss=0.0015 | lr=8.58e-05
  ✓ checkpoint sauvegardé (loss=0.0015)
[Pretrain] Epoch 007/20 | loss=0.0014 | lr=7.94e-05
  ✓ checkpoint sauvegardé (loss=0.0014)
[Pretrain] Epoch 008/20 | loss=0.0014 | lr=7.22e-05
  ✓ checkpoint sauvegardé (loss=0.0014)
[Pretrain] Epoch 009/20 | loss=0.0013 | lr=6.43e-05
  ✓ checkpoint sauvegardé (loss=0.0013)
[Pretrain] Epoch 010/20 | loss=0.0013 | lr=5.60e-05
  ✓ checkpoint sauvegardé (loss=0.0013)
[Pretrain] Epoch 011/20 | loss=0.0014 | lr=4.76e-05
[Pretrain] Epoch 012/20 | lo

## 6. Fine-tuning classification

In [14]:
from data import make_dataloaders
from finetune import finetune

cfg.epochs_finetune = FINETUNE_EPOCHS
cfg.batch_size_finetune = BATCH_SIZE_FINETUNE

train_dl, val_dl, test_dl = make_dataloaders(structures, labels, cfg)

train_labels = train_dl.dataset.labels

model = SupraJEPA(cfg)
model = finetune(model, train_dl, val_dl, test_dl, train_labels, cfg, device)

✓ Poids prétrainés chargés depuis checkpoints/jepa_pretrained.pt
  pos_weight calculé : 0.9  (n_pos=72, n_neg=68)


/content/SUPRACONDUCTOR-JEPA/model.py:111: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
/content/SUPRACONDUCTOR-JEPA/model.py:148: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


[Finetune] Epoch 001/80 | loss=1.1255 | AUROC=0.7867 | F1=0.0000 | TNR=1.0000 | TPR=0.0000
  ✓ checkpoint sauvegardé (AUROC=0.7867, F1=0.0000, TNR=1.0000)
[Finetune] Epoch 002/80 | loss=0.9727 | AUROC=0.9244 | F1=0.8000 | TNR=0.6000 | TPR=0.9333
  ✓ checkpoint sauvegardé (AUROC=0.9244, F1=0.8000, TNR=0.6000)
[Finetune] Epoch 003/80 | loss=0.7162 | AUROC=0.9556 | F1=0.8108 | TNR=0.5333 | TPR=1.0000
  ✓ checkpoint sauvegardé (AUROC=0.9556, F1=0.8108, TNR=0.5333)
[Finetune] Epoch 004/80 | loss=0.7411 | AUROC=0.9822 | F1=0.6957 | TNR=1.0000 | TPR=0.5333
  ✓ checkpoint sauvegardé (AUROC=0.9822, F1=0.6957, TNR=1.0000)
[Finetune] Epoch 005/80 | loss=0.8709 | AUROC=0.9822 | F1=0.9333 | TNR=0.9333 | TPR=0.9333
  ✓ checkpoint sauvegardé (AUROC=0.9822, F1=0.9333, TNR=0.9333)
[Finetune] Epoch 006/80 | loss=0.5489 | AUROC=0.9822 | F1=0.8333 | TNR=0.6000 | TPR=1.0000
[Finetune] Epoch 007/80 | loss=0.5911 | AUROC=0.9822 | F1=0.9032 | TNR=0.8667 | TPR=0.9333
[Finetune] Epoch 008/80 | loss=0.5516 | AUR

## 7. Diffusion generative + screening JEPA

Le DDPM apprend a generer des candidats dans l'espace token cristallin du JEPA : groupe d'espace, maille, elements, Wyckoff, coordonnees fractionnelles et presence des sites. Ensuite, chaque candidat est passe dans `SupraJEPA.forward_classify` pour obtenir `P(supraconducteur)`.


In [19]:
from diffusion import CrystalDDPM, batch_to_diffusion_vector, diffusion_vector_to_batch
from train_diffusion import train_diffusion

cfg.epochs_diffusion = DIFFUSION_EPOCHS
cfg.batch_size_diffusion = BATCH_SIZE_DIFFUSION

if TRAIN_DIFFUSION_ON_POSITIVES_ONLY:
    diffusion_pairs = [(s, y) for s, y in zip(structures, labels) if y == 1]
    diffusion_structures = [s for s, _ in diffusion_pairs]
    diffusion_labels = [y for _, y in diffusion_pairs]
    print('Diffusion entrainee sur SC uniquement:', len(diffusion_labels))
else:
    diffusion_structures = structures
    diffusion_labels = labels
    print('Diffusion entrainee sur toutes les structures:', len(diffusion_labels))

diffusion_ds = CrystalDataset(diffusion_structures, diffusion_labels, cfg.max_atoms, mask_ratio=None)
diffusion_loader = DataLoader(
    diffusion_ds,
    batch_size=cfg.batch_size_diffusion,
    shuffle=True,
    num_workers=2,
)

diffusion_model = CrystalDDPM(cfg)
diffusion_model = train_diffusion(diffusion_model, diffusion_loader, cfg, device)


Diffusion entrainee sur SC uniquement: 100
[Diffusion] Epoch 001/500 | loss=1.1337
  checkpoint sauvegardé (loss=1.1337)
[Diffusion] Epoch 002/500 | loss=1.0944
  checkpoint sauvegardé (loss=1.0944)
[Diffusion] Epoch 003/500 | loss=1.0639
  checkpoint sauvegardé (loss=1.0639)
[Diffusion] Epoch 004/500 | loss=1.0813
[Diffusion] Epoch 005/500 | loss=1.0835
[Diffusion] Epoch 006/500 | loss=1.0741
[Diffusion] Epoch 007/500 | loss=1.0535
  checkpoint sauvegardé (loss=1.0535)
[Diffusion] Epoch 008/500 | loss=1.0544
[Diffusion] Epoch 009/500 | loss=1.0405
  checkpoint sauvegardé (loss=1.0405)
[Diffusion] Epoch 010/500 | loss=1.0145
  checkpoint sauvegardé (loss=1.0145)
[Diffusion] Epoch 011/500 | loss=1.0420
[Diffusion] Epoch 012/500 | loss=1.0298
[Diffusion] Epoch 013/500 | loss=1.0219
[Diffusion] Epoch 014/500 | loss=1.0278
[Diffusion] Epoch 015/500 | loss=1.0426
[Diffusion] Epoch 016/500 | loss=1.0174
[Diffusion] Epoch 017/500 | loss=1.0109
  checkpoint sauvegardé (loss=1.0109)
[Diffusion]

## 8. Generer et classer de nouveaux candidats

Cette cellule sample le modele de diffusion, transforme les samples en batch JEPA, puis trie les candidats par probabilite de supraconductivite. Le JSON produit est une representation tokenisee lisible ; il faut ensuite ajouter une validation chimique/cristallographique avant toute interpretation physique forte.

Si tu observes surtout `H`/`Og`, des coordonnees `0/1`, ou des mailles extremes, le checkpoint diffusion est mauvais ou trop peu entraine. Re-entraine la section diffusion apres la correction de normalisation `bounded_features_v2`.


In [20]:
import json
from pymatgen.core import Element

IDX_TO_ELEMENT = {el.Z: el.symbol for el in Element}
IDX_TO_WYCKOFF = {idx + 1: letter for idx, letter in enumerate('abcdefghijklmnopqrstuvwxyz')}

@torch.no_grad()
def score_generated_candidates(jepa_model, diffusion_model, cfg, n_samples, top_k):
    diffusion_model.eval()
    jepa_model.eval()

    vectors = diffusion_model.sample(n_samples, device=device)
    batch = diffusion_vector_to_batch(vectors, cfg, min_atoms=1)
    batch = {k: v.to(device) for k, v in batch.items()}

    probs = torch.sigmoid(jepa_model.forward_classify(batch))
    order = torch.argsort(probs, descending=True)[:min(top_k, n_samples)]
    return batch, probs, order


def candidate_to_record(batch, probs, idx):
    valid = batch['padding_mask'][idx, 1:].detach().cpu()
    element_ids = batch['element_ids'][idx, 1:].detach().cpu()
    wyckoff_ids = batch['wyckoff_ids'][idx, 1:].detach().cpu()
    coords = batch['frac_coords'][idx, 1:].detach().cpu()
    lattice = batch['lattice_feat'][idx].detach().cpu()

    sites = []
    for site_idx in torch.where(valid)[0].tolist():
        z = int(element_ids[site_idx])
        wyck = int(wyckoff_ids[site_idx])
        sites.append({
            'element': IDX_TO_ELEMENT.get(z, f'Z{z}'),
            'Z': z,
            'wyckoff': IDX_TO_WYCKOFF.get(wyck, 'a'),
            'frac_coords': [float(x) for x in coords[site_idx].tolist()],
        })

    return {
        'p_superconductor': float(probs[idx].detach().cpu()),
        'space_group': int(batch['sg_id'][idx].detach().cpu()),
        'lattice': {
            'a': float(lattice[0] * 20.0),
            'b': float(lattice[1] * 20.0),
            'c': float(lattice[2] * 20.0),
            'alpha': float(lattice[3] * 180.0),
            'beta': float(lattice[4] * 180.0),
            'gamma': float(lattice[5] * 180.0),
        },
        'n_sites': len(sites),
        'sites': sites,
    }

batch_gen, probs_gen, order_gen = score_generated_candidates(
    model,
    diffusion_model,
    cfg,
    n_samples=N_GENERATED,
    top_k=TOP_K,
)

records = [candidate_to_record(batch_gen, probs_gen, int(idx)) for idx in order_gen]

with open('generated_candidates.json', 'w') as f:
    json.dump(records, f, indent=2)

print('Top candidats generes:')
for rank, rec in enumerate(records[:10], start=1):
    print(f"#{rank:02d} P(SC)={rec['p_superconductor']:.4f} SG={rec['space_group']} sites={rec['n_sites']}")

records[:3]


Top candidats generes:
#01 P(SC)=0.0140 SG=230 sites=29
#02 P(SC)=0.0014 SG=1 sites=25
#03 P(SC)=0.0003 SG=230 sites=28
#04 P(SC)=0.0002 SG=1 sites=32
#05 P(SC)=0.0002 SG=167 sites=28
#06 P(SC)=0.0002 SG=1 sites=23
#07 P(SC)=0.0001 SG=1 sites=29
#08 P(SC)=0.0001 SG=230 sites=32
#09 P(SC)=0.0001 SG=1 sites=33
#10 P(SC)=0.0001 SG=1 sites=34


[{'p_superconductor': 0.014025034382939339,
  'space_group': 230,
  'lattice': {'a': 50.0,
   'b': 1.0,
   'c': 1.0,
   'alpha': 180.0,
   'beta': 27.000001907348633,
   'gamma': 180.0},
  'n_sites': 29,
  'sites': [{'element': 'H',
    'Z': 1,
    'wyckoff': 'z',
    'frac_coords': [0.0, 1.0, 0.0]},
   {'element': 'H', 'Z': 1, 'wyckoff': 'z', 'frac_coords': [1.0, 0.0, 0.0]},
   {'element': 'Og', 'Z': 118, 'wyckoff': 'a', 'frac_coords': [0.0, 0.0, 1.0]},
   {'element': 'H', 'Z': 1, 'wyckoff': 'z', 'frac_coords': [0.0, 0.0, 0.0]},
   {'element': 'Og', 'Z': 118, 'wyckoff': 'a', 'frac_coords': [1.0, 1.0, 0.0]},
   {'element': 'Og', 'Z': 118, 'wyckoff': 'a', 'frac_coords': [1.0, 0.0, 0.0]},
   {'element': 'H', 'Z': 1, 'wyckoff': 'z', 'frac_coords': [1.0, 1.0, 0.0]},
   {'element': 'H', 'Z': 1, 'wyckoff': 'z', 'frac_coords': [0.0, 1.0, 1.0]},
   {'element': 'H', 'Z': 1, 'wyckoff': 'a', 'frac_coords': [0.0, 0.0, 0.0]},
   {'element': 'Og', 'Z': 118, 'wyckoff': 'a', 'frac_coords': [1.0, 1.0, 

## 9. Sauvegarder les artefacts


In [17]:
!ls -lh checkpoints data
!test -f generated_candidates.json && ls -lh generated_candidates.json || true

# Optionnel: sauvegarder dans Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/SUPRA-JEPA
# !cp -r checkpoints data /content/drive/MyDrive/SUPRA-JEPA/
# !test -f generated_candidates.json && cp generated_candidates.json /content/drive/MyDrive/SUPRA-JEPA/


checkpoints:
total 89M
-rw-r--r-- 1 root root 6.9M Jun 26 09:42 crystal_diffusion.pt
-rw-r--r-- 1 root root  42M Jun 26 09:42 jepa_finetune.pt
-rw-r--r-- 1 root root  42M Jun 26 09:40 jepa_pretrained.pt
-rw-r--r-- 1 root root   97 Jun 26 09:42 thresholds.json

data:
total 872K
-rw-r--r-- 1 root root 545K Jun 26 09:39 nonsc_structures.json
-rw-r--r-- 1 root root 321K Jun 26 09:39 supercon_structures.json
-rw-r--r-- 1 root root 105K Jun 26 09:42 generated_candidates.json


## Notes

- Le matching SuperCon -> Materials Project utilise des formules candidates exactes puis arrondies, car beaucoup de compositions experimentales sont dopees ou non stoechiometriques.
- Pour un vrai entrainement, augmente `MAX_SC`, `MAX_NONSC`, `PRETRAIN_EPOCHS`, `FINETUNE_EPOCHS` et `DIFFUSION_EPOCHS`.
- `TRAIN_DIFFUSION_ON_POSITIVES_ONLY=True` biaise le generateur vers les supraconducteurs connus ; `False` apprend une distribution plus generale de structures.
- Les candidats generes doivent etre filtres : distances minimales, elements realistes, compatibilite groupe d'espace/Wyckoff, puis relaxation DFT.
- Sur CPU, garde des petits batchs. Sur GPU Colab, tu peux monter progressivement.
